# Tier 2 HCOs & their HCPs

In [0]:
%sql
-- ============================================
-- Create a temporary view containing ALL treated patients
-- Treatment is identified via:
--   1) Medical claims with relevant NDCs
--   2) Pharmacy claims with relevant NDCs (PAID only)
--   3) Medical claims with infusion / procedure codes
-- Date window applied AFTER union to standardize cohort timing
-- ============================================

CREATE OR REPLACE TEMPORARY VIEW mpsii_treatment_table AS

SELECT *
FROM (

    -- --------------------------------------------
    -- MEDICAL EVENTS: Treatment identified via NDC
    -- Uses Rendering NPI if available, else Referring NPI
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION

    -- --------------------------------------------
    -- PHARMACY EVENTS: Treatment identified via NDC
    -- Includes only PAID transactions to ensure true utilization
    -- Prescriber NPI used as treating provider
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE, -- Not applicable for pharmacy claims
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION

    -- --------------------------------------------
    -- MEDICAL EVENTS: Treatment identified via
    -- infusion / administration / transplant procedure codes
    -- --------------------------------------------
    SELECT DISTINCT
        PATIENT_ID AS PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'J1743','S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242',
        '38243','38250'
    )

)

-- --------------------------------------------
-- Apply treatment window filter
-- Ensures consistent cohort timing across all sources
-- --------------------------------------------
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30';


In [0]:
%sql
-- =====================================================
-- Create a temporary view to calculate treated patients
-- at the HCO level (non-unique across time/events)
-- Provider attribution is rolled up from HCP → HCO
-- =====================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_non_unique AS

-- ---------------------------------------------
-- Step 1: Attach HCO NPI to treatment claims
-- Mapping is done via HCP NPI → HCO NPI reference
-- ---------------------------------------------
WITH t1 AS (
    SELECT
        a.*,               -- All treatment claim attributes
        b.hco_npi          -- Parent HCO mapped from HCP
    FROM mpsii_treatment_table AS a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0109 AS b
        ON a.npi = b.hcp_npi
)

-- ---------------------------------------------
-- Step 2: Aggregate treated patients at HCO level
-- Counts DISTINCT patients treated at each HCO
-- ---------------------------------------------
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS tx_patients_non_unique
FROM t1

-- ---------------------------------------------
-- Filters:
-- 1) Exclude invalid / placeholder HCO NPIs
-- 2) Ensure provider attribution exists
-- ---------------------------------------------
WHERE hco_npi != '-'
  AND npi IS NOT NULL

-- ---------------------------------------------
-- Final aggregation & ranking
-- ---------------------------------------------
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
%sql
-- ============================================================
-- Create UNIQUE treated patient counts at HCO level
-- Each patient is assigned to ONE primary treating HCP
-- HCP selection is based on:
--   1) Specialty priority
--   2) Number of visits
--   3) Most recent treatment date
-- ============================================================

CREATE OR REPLACE TEMPORARY VIEW tx_claims_unique AS

-- ============================================================
-- STEP 0: Base treatment population
-- ============================================================
WITH elaprase_treated_v1 AS (
    SELECT *
    FROM mpsii_treatment_table
),

-- ============================================================
-- STEP 1: Attach provider specialty & classify into buckets
-- Specialty hierarchy supports rare disease treatment logic
-- ============================================================
pulling_specialities AS (
    SELECT
        a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,

        -- --------------------------------------------
        -- Normalize specialties into analytical buckets
        -- --------------------------------------------
        CASE
            WHEN primary_specialty LIKE '%Genetic%'
              OR secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'

            WHEN primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'

            WHEN primary_specialty LIKE '%Psychiatry & Neurology%'
              OR secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR primary_specialty LIKE '%Neurological Surgery%'
              THEN 'Psychiatry & Neurology'

            WHEN primary_specialty LIKE '%Nurse Practitioner%'
              OR primary_specialty LIKE '%Physician Assistant%'
              THEN 'NPPA'

            WHEN primary_specialty LIKE '%Internal Medicine%'
              OR secondary_specialty LIKE '%Internal Medicine%'
              THEN 'PCP'

            WHEN primary_specialty LIKE '%Family Medicine%'
              OR secondary_specialty LIKE '%Family Medicine%'
              THEN 'PCP'

            WHEN a.npi IS NULL THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY

    FROM elaprase_treated_v1 a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.npi = b.npi
),

-- ============================================================
-- STEP 2: Assign ONE primary treating HCP per patient
-- Ranking logic:
--   Priority (specialty) →
--   Visit frequency →
--   Recency →
--   Stable NPI tie-break
-- ============================================================
elaprase_treated_v2 AS (

    SELECT DISTINCT
        n_pats AS patient_id,
        npi
    FROM (

        SELECT DISTINCT
            NPI,
            SPECIALTY,
            PATIENT_ID AS N_PATS,
            FILL_DATE,
            KH_PLAN,
            HCO_PRIMARY_NPI,
            PLACE_OF_SERVICE,
            'TX' AS PATIENT_TYPE
        FROM (

            SELECT DISTINCT
                PATIENT_ID,
                FILL_DATE,
                KH_PLAN,
                HCO_PRIMARY_NPI,
                PLACE_OF_SERVICE,
                NPI,
                SPECIALTY,
                FINAL_HCP_RANK
            FROM (

                -- --------------------------------------------
                -- Collapse multiple ranking layers
                -- --------------------------------------------
                SELECT *,
                       DENSE_RANK() OVER (
                           PARTITION BY PATIENT_ID
                           ORDER BY HCP_RANK_1
                       ) AS FINAL_HCP_RANK
                FROM (

                    SELECT *,
                           MIN(HCP_RANK) OVER (
                               PARTITION BY PATIENT_ID, NPI
                           ) AS HCP_RANK_1
                    FROM (

                        -- --------------------------------------------
                        -- Core HCP ranking logic
                        -- --------------------------------------------
                        SELECT *,
                               RANK() OVER (
                                   PARTITION BY PATIENT_ID
                                   ORDER BY
                                       PRIORITY,
                                       NO_OF_VISITS DESC,
                                       DATE(FILL_DATE) DESC,
                                       NPI
                               ) AS HCP_RANK
                        FROM (

                            -- --------------------------------------------
                            -- Precompute visit counts & specialty priority
                            -- --------------------------------------------
                            SELECT
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE,

                                -- Specialty priority (lower = higher priority)
                                CASE
                                    WHEN SPECIALTY = 'Geneticist' THEN 1
                                    WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2
                                    WHEN SPECIALTY = 'Pediatrician' THEN 3
                                    WHEN SPECIALTY = 'PCP' THEN 4
                                    WHEN SPECIALTY = 'NPPA' THEN 5
                                    WHEN SPECIALTY = 'Others' THEN 6
                                    ELSE 7
                                END AS PRIORITY,

                                COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                            FROM pulling_specialities
                            GROUP BY
                                PATIENT_ID,
                                NPI,
                                SPECIALTY,
                                FILL_DATE,
                                KH_PLAN,
                                HCO_PRIMARY_NPI,
                                PLACE_OF_SERVICE
                        )
                    )
                )
            )
            -- --------------------------------------------
            -- Retain only top-ranked HCP per patient
            -- --------------------------------------------
            WHERE HCP_RANK = 1
        )
    )
),

-- ============================================================
-- STEP 3: Filter valid HCP assignments
-- ============================================================
hcp_level AS (
    SELECT *
    FROM elaprase_treated_v2
    WHERE npi IS NOT NULL
),

-- ============================================================
-- STEP 4: Roll up unique patients from HCP → HCO
-- ============================================================
hco_level AS (
    SELECT
        a.*,
        b.hco_npi
    FROM hcp_level a
    LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 b
        ON a.npi = b.hcp_npi
)

-- ============================================================
-- STEP 5: Final UNIQUE treated patient count by HCO
-- Each patient contributes to exactly ONE HCO
-- ============================================================
SELECT
    hco_npi,
    COUNT(DISTINCT patient_id) AS tx_patients_unique
FROM hco_level
WHERE hco_npi != '-'
GROUP BY 1
ORDER BY 2 DESC;


In [0]:
%sql
SELECT * FROM tx_claims_unique ;